<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 2 章：处理文本数据

本 notebook 中使用的包：

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.9.1+rocm7.2.0.lw.git7e1940d4
tiktoken version: 0.13.0


- 本章涵盖数据准备和采样，使输入数据为 LLM "做好准备"

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

&nbsp;
## 2.1 理解词嵌入

- 本节没有代码

- 嵌入有多种形式；本书重点讨论文本嵌入

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLM 在高维空间（即数千个维度）中使用嵌入
- 由于我们无法可视化如此高维的空间（人类只能感知 1、2 或 3 个维度），下图展示了一个 2 维嵌入空间

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

&nbsp;
## 2.2 文本分词

- 在本节中，我们对文本进行分词，即将文本分解为更小的单元，例如单个单词和标点符号

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- 加载我们要处理的原始文本
- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) 是一篇公有领域的短篇小说

**下载逻辑说明：**
- `os.path.exists()` 检查文件是否已存在，只在文件不存在时才下载，避免重复请求
- `requests.get(url, timeout=30)` 发起 HTTP GET 请求，30 秒超时防止网络卡死
- `response.raise_for_status()` 检查 HTTP 状态码，404/500 等直接抛异常
- `with open(file_path, "wb") as f` 以二进制写入模式保存文件，`with` 确保写完后自动关闭
- 原书使用 `urllib.request.urlretrieve`，但该库的 SSL/TLS 协议设置较旧，部分使用 VPN 的读者会遇到 `ssl.SSLCertVerificationError` 错误，`requests` 库兼容性更好

In [ ]:
## 这段代码的功能是：下载训练用的文本数据集 the-verdict.txt（Edith Wharton 的短篇小说《The Verdict》）。
import os
import requests

file_path = "the-verdict.txt"
url = (
    "https://raw.githubusercontent.com/rasbt/"
    "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
    "the-verdict.txt"
)

if not os.path.exists(file_path):
    for attempt in range(1, 4):  # 最多重试 3 次
        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(response.content)
            print(f"已下载 {file_path}（{len(response.content):,} 字节）")
            break
        except requests.RequestException as e:
            print(f"第 {attempt} 次下载失败: {e}")
            if attempt == 3:
                raise RuntimeError(f"下载 {url} 失败，请检查网络后重试") from e
else:
    print(f"{file_path} 已存在，跳过下载")

the-verdict.txt 已存在，跳过下载


<br>

---

<br>

#### SSL 证书错误排查

- 一些读者报告在 VSCode 或 Jupyter 中运行 `urllib.request.urlretrieve` 时遇到 ssl.SSLCertVerificationError：`SSL: CERTIFICATE_VERIFY_FAILED`
- 这通常意味着 Python 的证书包已过时。


**解决方法**

- 使用 Python ≥ 3.9；你可以通过执行以下代码检查 Python 版本：
```python
import sys
print(sys.__version__)
```
- 升级证书包：
  - pip：`pip install --upgrade certifi`
  - uv：`uv pip install --upgrade certifi`
- 升级后重启 Jupyter 内核。
- 如果在执行上一个代码单元时仍然遇到 `ssl.SSLCertVerificationError`，请参阅 [GitHub 上的讨论](https://github.com/rasbt/LLMs-from-scratch/pull/403)

<br>

---

<br>

**读取文件代码分析：**

```python
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
```

- `open("the-verdict.txt", "r", encoding="utf-8")` — 以只读模式（`"r"`）打开文件，指定 UTF-8 编码
- `f.read()` — **一次性读取整个文件**为单个字符串（20,479 个字符）
- `with` 语句确保读取完毕后**自动关闭文件**，即使中途报错也会关闭

**为什么用 `with` 而不是直接 `open()`？**

```python
# 不推荐 — 忘记 close() 会泄漏文件描述符
f = open("the-verdict.txt", "r", encoding="utf-8")
raw_text = f.read()
f.close()

# 推荐 — 自动 close()
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
```

`raw_text` 变量会被后续所有 cell 使用：分词 → 构建 vocab → BPE 编码 → DataLoader → Embedding

In [4]:
# 读取训练数据集：Edith Wharton 短篇小说《The Verdict》
# 该文本将用于后续的分词、构建词汇表和 DataLoader
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"文件总字符数: {len(raw_text):,}")
print(f"前 99 个字符预览:\n{raw_text[:99]}")

文件总字符数: 20,479
前 99 个字符预览:
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- 目标是对这段文本进行分词和嵌入，以供 LLM 使用
- 让我们先基于一些简单的示例文本开发一个简单的分词器，然后再将其应用到上面的文本中

**分词器构建分四步逐步完善：**

| 步骤 | 正则 | 拆分依据 | 保留分隔符 |
|------|------|---------|-----------|
| 1 | `(\s)` | 仅空白 | 是（`()` 捕获分组） |
| 2 | `([,.]\|\s)` | 逗号 + 句号 + 空白 | 是 |
| 3 | — | 过滤空字符串 | — |
| 4 | 完整正则 | 所有标点 + 破折号 + 空白 | 是 |

- 下面的正则表达式是**步骤 1**：仅按空白字符拆分

In [6]:
# 步骤 1：按空白字符拆分文本
# re.split 中使用捕获分组 () 会保留分隔符本身（空白字符）
# 这是构建分词器的第一步，后续会进一步拆分标点
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- **步骤 2**：不仅按空白分割，还按逗号和句号分割

正则 `([,.]|\s)` 的含义：
- `[,.]` — 字符类，匹配逗号或句号
- `|\s` — 或空白字符
- `()` — 捕获分组，保留分隔符在结果中

**注意**：标点与空白相邻时（如 `', '`）会产生空字符串 `''`，下一步会过滤掉

In [7]:
# 步骤 2：同时按逗号、句号和空白字符拆分
# 正则含义: [,.] 匹配逗号或句号，|\s 匹配空白字符，| 表示"或"
# 注意：标点与空白相邻时会产生空字符串，下一步会过滤掉
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- **步骤 3**：过滤空字符串和纯空白项

`[item for item in result if item.strip()]` 的工作原理：
- `item.strip()` — 去除首尾空白，空字符串 `''` 和纯空格 `' '` 都返回 `""`（**falsy**）
- `if item.strip()` — 只保留 strip 后非空的项
- 逗号 `,`、句号 `.` 等标点不含空白，strip 后非空，因此被保留

| 输入 | strip() 结果 | falsy? | 保留? |
|------|-------------|--------|------|
| `'Hello'` | `'Hello'` | 否 | 是 |
| `','` | `','` | 否 | 是 |
| `''` | `''` | **是** | 否 |
| `' '` | `''` | **是** | 否 |

In [6]:
# 步骤 3：过滤空字符串和纯空白项
# item.strip() 去除首尾空白，空字符串和纯空格返回 ""（falsy），被过滤掉
# 逗号、句号等标点不含空白，strip() 后非空，因此被保留
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- **步骤 4**：使用完整正则表达式处理所有标点符号

正则 `([,.:;?_!"()\']|--|\s)` 的三部分：

```
[,.:;?_!"()']  → 匹配常见标点（逗号、句号、冒号、分号、问号、感叹号、引号、括号）
|               → 或
--              → 匹配双连字符（破折号/em-dash）
|               → 或
\s              → 匹配任何空白字符（空格、Tab、换行）
```

这一行相比步骤 2 多了 `[item.strip() for ...]`（左边有 `.strip()`）：既去除每项首尾空白，又过滤空串。这是**最终版分词逻辑**，后续直接应用到完整文本。

In [7]:
# 步骤 4：使用完整正则表达式拆分所有标点和空白
# 正则三部分: [,.:;?_!"()'] 匹配常见标点 | -- 匹配双连字符(破折号) | \s 匹配空白
# strip() 去除每项首尾空白，if item.strip() 过滤空串
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- 最终版分词器已验证通过，现在将其**应用到实际训练文本** `raw_text`（《The Verdict》全文，20,479 字符）
- 逻辑与步骤 4 完全相同，只是输入从测试字符串换成了完整文本
- 结果 `preprocessed` 包含 4,690 个 token，这些 token 将用于构建词汇表

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [8]:
# 将完整正则分词器应用到实际训练文本 raw_text（《The Verdict》全文）
# 逻辑与步骤 4 相同，只是从测试字符串换成了 20,479 字符的完整文本
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- 让我们计算 token 的总数

In [9]:
print(len(preprocessed))

4690


&nbsp;
## 2.3 将 token 转换为 token ID

- 接下来，我们将文本 token 转换为 token ID，以便后续通过嵌入层进行处理

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- 从这些 token 中，我们现在可以构建一个由所有唯一 token 组成的词汇表

In [10]:
# 构建词汇表：去重 → 排序 → 统计大小
# set() 去重（4,690 个 token → 1,130 个唯一值）
# sorted() 按字母排序，确保词汇表顺序固定、token→ID 映射一致
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [11]:
# 将排序后的唯一 token 列表转换为 {token: integer} 字典
# enumerate() 为每个 token 分配一个整数 ID（从 0 开始）
# 例如: {'!': 0, '"': 1, "'": 2, '(': 3, ...}
# 这个字典就是词汇表，后续 encode() 用它将文本转为 ID 序列
vocab = {token:integer for integer,token in enumerate(all_words)}

- 以下是该词汇表中的前 50 个条目：

In [12]:
# 预览词汇表前 51 条（ID 0~50）
# vocab.items() 返回 (token, integer) 对，因 sorted() 所以按字母序排列
# 输出格式: ('!', 0), ('"', 1), ("'", 2), ('(', 3), ...
# 可以看到标点排在前面（ASCII 码小），然后是大写单词，最后是小写单词
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- 下面我们使用一个小型词汇表来说明一段简短示例文本的分词过程：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- 现在将所有内容整合到一个分词器类中

In [13]:
# SimpleTokenizerV1: 最简分词器，支持 encode（文本→ID）和 decode（ID→文本）
#
# 数据流:
#   encode: "Hello, world" → ['Hello', ',', 'world'] → [1158, 5, 995]
#   decode: [1158, 5, 995] → ['Hello', ',', 'world'] → "Hello, world"
#
# 注意: V1 没有处理未知词（不在词汇表中的词会报 KeyError），V2 会解决
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # 正向映射: token → ID（直接使用传入的 vocab 字典）
        self.str_to_int = vocab
        # 反向映射: ID → token（字典推导式翻转 key/value）
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        # 步骤 1: 用完整正则拆分文本（与步骤 4 相同的正则）
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        # 步骤 2: strip 每项 + 过滤空串
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # 步骤 3: 查找 vocab 字典，将每个 token 转为整数 ID
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # 步骤 1: 将 ID 列表转回 token，用空格连接
        text = " ".join([self.int_to_str[i] for i in ids])
        # 步骤 2: 去掉标点前的多余空格（"Hello , " → "Hello, "）
        # \s+ 匹配标点前的一个或多个空白，r'\1' 替换为标点本身
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- `encode` 函数将文本转换为 token ID
- `decode` 函数将 token ID 转换回文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- 我们可以使用分词器将文本编码（即分词）为整数
- 这些整数随后可以被嵌入（稍后）作为 LLM 的输入

In [14]:
# 用 SimpleTokenizerV1 对实际文本进行 encode
# 输入: 包含引号、撇号、逗号、换行的复杂英文句子
# 输出: 整数 ID 列表 [1, 56, 2, 850, ...]，每个 ID 对应词汇表中的一个 token
# 这些整数 ID 后续会通过嵌入层（Embedding）转换为向量，作为 LLM 的输入
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- 我们可以将整数解码回文本

In [15]:
# decode: 将整数 ID 列表还原为文本
# 内部流程: ID → 查反向映射表得到 token → 空格连接 → 去掉标点前空格
# 注意: 还原结果与原始文本略有差异（如 It's 被拆为 It + ' + s），这是按词分词的固有局限
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [16]:
# 验证 encode→decode 往返（roundtrip）：文本 → ID → 文本
# 如果分词器实现正确，decode(encode(text)) 应能还原出可读的文本
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

&nbsp;
## 2.4 添加特殊上下文 token

- 添加一些"特殊" token 来处理未知单词和标记文本结尾是很有用的

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- 一些分词器使用特殊 token 来帮助 LLM 获得额外的上下文信息
- 一些特殊 token 包括
  - `[BOS]`（序列开头）标记文本的开始
  - `[EOS]`（序列结尾）标记文本的结束（通常用于拼接多个不相关的文本，例如两篇不同的维基百科文章或两本不同的书等）
  - `[PAD]`（填充）当我们以大于 1 的批量大小训练 LLM 时（我们可能包含不同长度的多个文本；使用填充 token 将较短的文本填充到最长长度，使所有文本长度相等）
- `[UNK]` 用于表示词汇表中未包含的单词

- 注意，GPT-2 不需要上述任何 token，它只使用一个 `|endoftext|` token 以降低复杂度
- `|endoftext|` 类似于上面提到的 `[EOS]` token
- GPT 也使用 `|endoftext|` 进行填充（因为我们在批量输入上训练时通常会使用掩码，所以不管填充 token 是什么都无所谓，我们不会关注被填充的 token）
- GPT-2 不使用 `<UNK>` token 处理词汇表外的单词；相反，GPT-2 使用字节对编码（BytePair encoding，BPE）分词器，它将单词分解为子词单元，我们将在后面的章节中讨论



- 我们在两个独立的文本源之间使用 `|endoftext|` token：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- 让我们看看对以下文本进行分词会发生什么：

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- 上述操作产生了错误，因为单词 "Hello" 不包含在词汇表中
- 为了处理这种情况，我们可以在词汇表中添加特殊 token（如 `"<|unk|>"`）来表示未知单词
- 既然我们已经在扩展词汇表，那我们再添加一个名为 `"|endoftext|"` 的 token，它在 GPT-2 训练中用于标记文本的结尾（也用于拼接文本之间，比如我们的训练数据集包含多篇文章、多本书等）

In [18]:
# 扩展词汇表：添加两个特殊 token
#   <|endoftext|> — 文本结束/分隔符，GPT-2 用它拼接多段独立文本（如不同文章）
#   <|unk|>     — 未知词标记，遇到词汇表外的词时不报错，而是映射为此 token
# 因为 sorted() 按字母序排列，这两个特殊 token 会排在最后（ID 1130 和 1131）
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["|endoftext|", "<|unk|>"])

# 重新构建词汇表字典（现在有 1,132 个条目）
vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [19]:
len(vocab.items())

1132

In [20]:
# 查看词汇表最后 5 条，验证特殊 token 已正确添加到末尾
# 预期输出: ('yourself', 1129), ('|endoftext|', 1130), ('<|unk|>', 1131)
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- 我们还需要相应地调整分词器，使其知道何时以及如何使用新的 `<unk>` token

In [21]:
# SimpleTokenizerV2: 在 V1 基础上增加未知词处理
#
# V1 vs V2 的区别:
#   V1: 遇到词汇表外的词 → KeyError 崩溃
#   V2: 遇到词汇表外的词 → 替换为 <|unk|> token，优雅处理
#
# 注意: GPT-2 实际上不需要 <|unk|>，因为 BPE 分词器能将任何未知词拆为子词
# 这里只是演示特殊 token 的概念，后续会用 tiktoken 替代这个简单分词器
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # 正向映射: token → ID
        self.str_to_int = vocab
        # 反向映射: ID → token
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        # 步骤 1: 正则分词 + strip + 过滤空串（与 V1 相同）
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # 步骤 2: 【V2 新增】检查每个 token 是否在词汇表中
        #   在词汇表中 → 保留原 token
        #   不在词汇表中 → 替换为 "<|unk|>"（ID 1131）
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        # 步骤 3: 查字典转为 ID（现在不会 KeyError 了）
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        # ID → token → 空格连接 → 去掉标点前空格
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

让我们使用修改后的分词器来对文本进行分词：

In [22]:
# 演示 <|endoftext|> 和 <|unk|> 两个特殊 token 的作用
tokenizer = SimpleTokenizerV2(vocab)

# text1 包含未知词 "Hello"（不在词汇表中，将被替换为 <|unk|>）
text1 = "Hello, do you like tea?"
# text2 包含未知词 "palace"（同样不在词汇表中）
text2 = "In the sunlit terraces of the palace."

# 用 <|endoftext|> 拼接两段独立文本
# 这是 GPT-2 训练时的标准做法：多篇文章/书籍之间用 <|endoftext|> 分隔
text = "<|endoftext|>".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [23]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [24]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

&nbsp;
## 2.5 字节对编码（BPE）

- GPT-2 使用字节对编码（BytePair encoding，BPE）作为其分词器
- 它允许模型将不在其预定义词汇表中的单词分解为更小的子词单元甚至单个字符，从而能够处理词汇表外的单词
- 例如，如果 GPT-2 的词汇表中没有单词 "unfamiliarword"，它可能会将其分词为 ["unfam", "iliar", "word"] 或其他子词分解方式，具体取决于其训练好的 BPE 合并规则
- 原始 BPE 分词器可以在这里找到：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 在本章中，我们使用 OpenAI 开源的 [tiktoken](https://github.com/openai/tiktoken) 库中的 BPE 分词器，该库用 Rust 实现了核心算法以提高计算性能
- 我在 [./bytepair_encoder](../02_bonus_bytepair-encoder) 中创建了一个 notebook，并排比较了这两种实现（tiktoken 在示例文本上大约快 5 倍）

In [25]:
# pip install tiktoken

In [26]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.7.0


In [27]:
tokenizer = tiktoken.get_encoding("gpt2")

In [28]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [29]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE 分词器将未知单词分解为子词和单个字符：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 滑动窗口数据采样

- 我们训练 LLM 每次生成一个单词，因此我们希望相应地准备训练数据，其中序列中的下一个单词就是预测目标：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [30]:
# 用 BPE 分词器（tiktoken gpt2）对完整文本进行编码
# 对比: 简单分词器产生 4,690 个 token，BPE 产生 5,145 个 token
# BPE 更多是因为它将标点（如 em-dash --）拆得更细，但能处理任意未知词
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- 对于每个文本块，我们需要输入和目标
- 由于我们希望模型预测下一个单词，目标是将输入向右移动一个位置

In [31]:
# 截取第 50 个 token 之后的部分，跳过文本开头的通用 token
# 用作后续滑动窗口的示例数据，展示输入-目标对的构造方式
# 后续会用这个数据演示: x=[290,4920,2241,287] → y=[4920,2241,287,257]（右移 1 位）
enc_sample = enc_text[50:]

In [32]:
# 滑动窗口大小（上下文长度）：一次看 4 个 token
context_size = 4

# x: 输入序列，取前 context_size 个 token  → [290, 4920, 2241, 287]
# y: 目标序列，右移 1 位（每个位置 = 下一个 token）→ [4920, 2241, 287, 257]
# 直觉：给定 x[i]，模型应该预测 y[i]（即 x[i+1]）
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- 逐个地，预测过程如下所示：

In [33]:
# 逐位展开滑动窗口，展示"下一个 token 预测"的训练目标
# i=1: [290]           → 4920   （1 个 token 预测第 2 个）
# i=2: [290, 4920]     → 2241   （2 个 token 预测第 3 个）
# i=3: [290, 4920, 2241] → 287   （3 个 token 预测第 4 个）
# i=4: [290, 4920, 2241, 287] → 257 （4 个 token 预测第 5 个）
for i in range(1, context_size+1):
    context = enc_sample[:i]      # 输入：前 i 个 token（逐位增长）
    desired = enc_sample[i]       # 目标：第 i+1 个 token

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [34]:
# 与上一个 cell 逻辑相同，但用 tokenizer.decode 将 token ID 还原为可读文本
# 这样能直观看到"下一个词预测"在自然语言层面是什么效果
# 输出示例:
#   " and" ----> " established"
#   " and established" ----> " himself"
#   " and established himself" ----> " in"
#   " and established himself in" ----> " a"
# 注意: desired 是单个 int，decode() 需要列表，所以传入 [desired]
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- 我们将在后续章节中介绍注意力机制后再处理下一个词预测
- 现在，我们实现一个简单的数据加载器，它遍历输入数据集并返回移动了一位的输入和目标

- 安装并导入 PyTorch（安装提示请参见附录 A）

In [35]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.5.1


- 我们使用滑动窗口方法，每次将位置移动 +1：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/13.webp?123" width="500px">

- 创建数据集和数据加载器，从输入文本数据集中提取文本块

In [36]:
from torch.utils.data import Dataset, DataLoader


# GPTDatasetV1: 将原始文本转换为 PyTorch Dataset，用滑动窗口生成输入-目标对
# 继承 Dataset 后需实现 __len__ / __getitem__，DataLoader 才能自动 batch、shuffle
#
# 参数说明:
#   txt        — 原始文本字符串
#   tokenizer  — BPE 分词器（tiktoken GPT-2）
#   max_length — 每个样本的 token 数（即上下文窗口大小）
#   stride     — 窗口每次滑动的步长；stride = max_length 时无重叠，stride < max_length 时有重叠
#
# 数据流: txt → encode → [token1, token2, ...] → 滑动窗口切片 → (input_chunk, target_chunk) 对
#   input_chunk  = token_ids[i      : i+max_length]     ← 输入序列
#   target_chunk = token_ids[i+1    : i+max_length+1]   ← 目标序列（整体右移 1 位）
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []    # 存放所有输入 chunk
        self.target_ids = []   # 存放所有目标 chunk（与 input_ids 一一对应）

        # 第 1 步：将整篇文本一次性编码为 token ID 列表
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # 确保文本足够长，至少能切出一个完整的窗口
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # 第 2 步：滑动窗口切分 — 每次前进 stride 步，切出 max_length 长度的重叠子序列
        # range 上界为 len(token_ids) - max_length，确保最后一个窗口不会越界
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i : i + max_length]            # 输入：从 i 开始取 max_length 个 token
            target_chunk = token_ids[i + 1 : i + max_length + 1]  # 目标：从 i+1 开始取 max_length 个 token（右移 1 位）
            self.input_ids.append(torch.tensor(input_chunk))       # 转为 PyTorch tensor 存储
            self.target_ids.append(torch.tensor(target_chunk))

    # 返回数据集中的样本数量（有多少个滑动窗口）
    def __len__(self):
        return len(self.input_ids)

    # 按索引取出一个 (input, target) 对 — DataLoader 内部会调用此方法
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [37]:
# create_dataloader_v1: 封装「文本 → Dataset → DataLoader」的完整流程
# 一步到位地完成: 分词 → 滑动窗口切片 → batch 化 → shuffle，返回可直接迭代的 DataLoader
#
# 参数说明:
#   txt         — 原始文本字符串
#   batch_size  — 每个 batch 包含多少个样本（默认 4）
#   max_length  — 每个样本的 token 数 / 上下文窗口大小（默认 256）
#   stride      — 滑动窗口步长（默认 128）；stride < max_length 时窗口之间有重叠，样本更多
#   shuffle     — 是否打乱样本顺序（默认 True）
#   drop_last   — 是否丢弃最后一个不完整的 batch（默认 True，保证所有 batch 大小一致）
#   num_workers — 多进程数据加载的进程数（默认 0 = 主进程加载）
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # 第 1 步：初始化 BPE 分词器（GPT-2 编码方案）
    tokenizer = tiktoken.get_encoding("gpt2")

    # 第 2 步：用 GPTDatasetV1 将文本切分为滑动窗口样本（input, target 对）
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 第 3 步：用 DataLoader 包装 Dataset，实现自动 batch 化、shuffle、多进程加载
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- 让我们以批量大小为 1 测试数据加载器，用于上下文大小为 4 的 LLM：

In [38]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [39]:
# 用小参数测试 DataLoader：batch_size=1, max_length=4, stride=1（每次只滑 1 步，窗口高度重叠）
# shuffle=False 保持原始顺序，方便观察滑动窗口的切分过程
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

# iter() 将 DataLoader 转为迭代器，next() 取出第一个 batch
# first_batch 是一个列表: [input_tensor, target_tensor]
#   input_tensor:  shape (1, 4) → [[  40,  367, 2885, 1464]]
#   target_tensor: shape (1, 4) → [[ 367, 2885, 1464, 1807]]  （右移 1 位）
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [40]:
# 取第二个 batch，观察 stride=1 的效果：窗口只右移了 1 步
# 第 1 batch: input [[  40,  367, 2885, 1464]] → target [[ 367, 2885, 1464, 1807]]
# 第 2 batch: input [[ 367, 2885, 1464, 1807]] → target [[2885, 1464, 1807, 3619]]
#          ↑ 正好是第 1 batch 的 target！stride=1 时连续窗口高度重叠
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


**Batch 1 vs Batch 2 对比分析（stride=1 的效果）：**

```
token 序列:  [40, 367, 2885, 1464, 1807, 3619, ...]

Batch 1:     [40, 367, 2885, 1464 | 1807, 3619, ...]
              └──── input ────┘     └ target(右移1) ┘

Batch 2:          [367, 2885, 1464, 1807 | 3619, ...]
                   └──── input ────┘     └ target ┘
```

| | Batch 1 | Batch 2 |
|---|---|---|
| **起始位置** | token[0] | token[1] |
| **input** | `[40, 367, 2885, 1464]` | `[367, 2885, 1464, 1807]` |
| **target** | `[367, 2885, 1464, 1807]` | `[2885, 1464, 1807, 3619]` |
| **关系** | — | Batch 2 的 **input = Batch 1 的 target** |

关键点：
- **只新进来 1 个 token**（`3619`），丢弃最前面 1 个（`40`）
- **重叠 = max_length - stride = 4 - 1 = 3 个 token**
- Batch 2 的 input 就是 Batch 1 的 target，因为 target 本身就是 input 右移 1 位

> **实践建议**：stride 太小（如 1）会导致大量重叠样本，数据冗余、训练慢。生产中通常设 `stride = max_length`（无重叠）或 `stride = max_length / 2`（适度重叠）。

- 下面是一个使用步长等于上下文长度（此处为 4）的示例：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- 我们也可以创建批量输出
- 注意，我们在这里增加了步长，以避免批次之间的重叠，因为更多的重叠可能导致过拟合加剧

In [41]:
# stride = max_length = 4：窗口之间无重叠，每个 token 只出现在一个样本中（避免数据冗余）
# batch_size=8：每个 batch 包含 8 个样本
# shuffle=False：保持顺序，方便观察
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
# 解包：DataLoader 返回的每个 batch 是 (inputs, targets) 元组
#   inputs:  shape (8, 4) — 8 个样本 × 4 个 input token
#   targets: shape (8, 4) — 8 个样本 × 4 个 target token（整体右移 1 位）
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


**参数组合分析：`stride=4, max_length=4, batch_size=8, shuffle=False`**

**1. `stride = max_length = 4` → 窗口无重叠**

```
token 序列:  [40, 367, 2885, 1464 | 1807, 3619, 402, 271 | ...]

窗口 0:      [40, 367, 2885, 1464]           ← token[0:4]
窗口 1:                           [1807, 3619, 402, 271]  ← token[4:8]
窗口 2:                                                    [...]
                                 ↑ 无任何重叠
```

| stride | 重叠 | 样本数量 | 数据冗余 |
|---|---|---|---|
| 1 | 3 个 token | 多 | 高（同一段文本被反复学习） |
| 4 (= max_length) | 0 | 少 | 无 |
| 2 (= max_length/2) | 2 个 token | 适中 | 适度 |

**2. `batch_size=8` → 每个 batch 包含 8 个样本**

```
inputs:  shape (8, 4)  ← 8 行 × 4 列
         sample 0: [40, 367, 2885, 1464]
         sample 1: [1807, 3619, 402, 271]
         ...
         sample 7: [...]
```

**3. `shuffle=False` → 保持原始顺序**

不打乱样本顺序，连续两个 batch 的文本是紧挨着的。此处设 False 是为了观察切分过程；实际训练时设 `True`，避免模型记住文本顺序。

> **三者组合效果**：把整篇文本按 4 个 token 一段、无重叠地切开，每 8 段打包成一个 batch，按原始顺序依次输出。

&nbsp;
## 2.7 创建 token 嵌入

- 数据已经基本为 LLM 准备就绪
- 但最后，让我们使用嵌入层将 token 嵌入到连续的向量表示中
- 通常，这些嵌入层是 LLM 本身的一部分，并在模型训练期间被更新（训练）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- 假设我们有以下四个输入示例，输入 ID 分别为 2、3、5 和 1（分词后）：

In [42]:
input_ids = torch.tensor([2, 3, 5, 1])

- 为简单起见，假设我们有一个仅包含 6 个单词的小型词汇表，并且我们想创建大小为 3 的嵌入：

In [43]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 这将产生一个 6x3 的权重矩阵：

In [44]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- 对于熟悉 one-hot 编码的人来说，上面的嵌入层方法本质上只是一种更高效的实现方式，等效于 one-hot 编码后接全连接层中的矩阵乘法，这在补充代码 [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul) 中有详细说明
- 由于嵌入层只是一种等效于 one-hot 编码和矩阵乘法方法的更高效实现，它可以被视为一个可以通过反向传播进行优化的神经网络层

- 要将 ID 为 3 的 token 转换为 3 维向量，我们执行以下操作：

In [45]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- 注意，上面的是 `embedding_layer` 权重矩阵的第 4 行
- 要嵌入上面所有四个 `input_ids` 值，我们执行

In [46]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- 嵌入层本质上是一个查找操作：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **你可能对比较嵌入层与普通线性层的补充内容感兴趣：[../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

&nbsp;
## 2.8 编码词位置

- 嵌入层将 ID 转换为相同的向量表示，无论它们在输入序列中的位置如何：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- 位置嵌入与 token 嵌入向量相结合，形成大型语言模型的输入嵌入：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- BytePair 编码器的词汇表大小为 50,257：
- 假设我们想将输入 token 编码为 256 维向量表示：

In [47]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- 如果我们从数据加载器中采样数据，我们将每个批次中的 token 嵌入到 256 维向量中
- 如果批量大小为 8，每个批次有 4 个 token，这将产生一个 8 x 4 x 256 的张量：

In [48]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [49]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [50]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


- GPT-2 使用绝对位置嵌入，因此我们只需创建另一个嵌入层：

In [51]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

In [52]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


- 要创建 LLM 中使用的输入嵌入，我们只需将 token 嵌入和位置嵌入相加：

In [53]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


- 在输入处理流程的初始阶段，输入文本被分割为单独的 token
- 分割后，这些 token 根据预定义的词汇表转换为 token ID：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

&nbsp;
## 总结与要点

参见 [./dataloader.ipynb](./dataloader.ipynb) 代码 notebook，它是我们在本章实现的数据加载器的精简版本，后续章节训练 GPT 模型时会用到。

练习解答请参见 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)。

如果你有兴趣了解如何从头实现和训练 GPT-2 分词器，请参阅 [Byte Pair Encoding (BPE) Tokenizer From Scratch](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) notebook。